<a href="https://colab.research.google.com/github/AllenInstitute/SPOCTT/blob/main/ABCAtlas_UseCases/SEAAD_MTG_RNAseq_SST_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="title"></a>
# Seattle Alzheimer's Disease Brain Cell Atlas (SEA-AD) MTG RNAseq Exploration

**Seattle Alzheimer's Disease Brain Cell Atlas** (SEA-AD) Middle Temporal Gyrus (MTG) snRNAseq

Direct data source: https://sea-ad-single-cell-profiling.s3.amazonaws.com/index.html#MTG/RNAseq/

Data source w/info: https://registry.opendata.aws/allen-sea-ad-atlas/

Publication: Gabitto, Travaglini, et al., *Nature Neuroscience* 2024: https://www.nature.com/articles/s41593-024-01774-5


**<font color="red" size=4> IMPORTANT: This notebook is READ-ONLY. To edit and run this notebook, follow the steps below.</font>**


---
<a name="save"></a>
## Before you start - Save this notebook!

When you open a new Colab notebook from the WebCampus (like you hopefully did for this one), you cannot save changes. So it's  best to store the Colab notebook in your personal drive `"File > Save a copy in Drive..."` **before** you do anything else.

The file will open in a new tab in your web browser, and it is automatically named something like: "**Copy of SEAAD_MTG_RNAseq_SST.ipynb**". <br>You can rename this to just the title "**SEAAD_MTG_RNAseq_SST_Notebook.pynb**".<br>
Make sure you do keep an informative name so that you know what the file is.


___

**Where does the notebook get saved in Google Drive?**

By default, the notebook will be copied to a folder called `Colab Notebooks` at the root (home directory) of your Google Drive.

___

<a name="audience"></a>
# Target audience for this notebook

This notebook walks through exploring the SEA-AD middle temporal gyrus (MTG) single-nucleus RNA-seq data, from downloading it out of the public AWS bucket to asking a real question from the paper: **are somatostatin (Sst) interneurons affected by Alzheimer's disease pathology?**

It is written to be **beginner-friendly**, but anyone can use and adapt it, whatever your level of experience.

**If you're new to Python, Colab or single-cell data:**
- Start with the [Intro to Colab](#colab-intro), then read [What is snRNA-seq?](#intro-snrnaseq), [About the Dataset](#intro-dataset) and [The AnnData object](#intro-anndata).
- Run each cell in order (click the ▶ button or press `Shift + Enter`), and only change lines marked `**EDIT**`.
- [Parts 1](#part-1) and [2](#part-2) (getting the data, exploring metadata, QC and UMAPs) are the most beginner-friendly.
- [Parts 3](#part-3) and [4](#part-4) get more advanced (extracting a subset from the large expression matrix, differential expression, pseudobulk and volcano plots). You don't need to follow every line of code. Run the cells and focus on the plots and the interpretation sections.
- [Part 5](#part-5) has no code provided. It's a chance to adapt what you've learned to a different cell type, at whatever pace suits you.

**If you're experienced:** use it as a working template. The data access, memory-friendly subsetting, and analysis steps can be copied into your own pipeline, and Part 5 outlines how to repeat the whole workflow for any other subclass.

**What you'll learn and end up with:**
- How to find, download and open SEA-AD data
- How to explore cell metadata, AD pathology measures and basic quality control
- An Sst-only AnnData object, which you can save to skip the slow steps next time
- Plots comparing Sst supertypes across AD pathology, connected back to the findings in the paper

**What this notebook doesn't do:** it isn't a full course in single-cell methods or Alzheimer's biology, and the analyses are simplified for teaching (see the interpretation notes for caveats). For the full picture, see the [publication](https://www.nature.com/articles/s41593-024-01774-5).

**What you need:**
- A Google account (to run Colab and, optionally, save to Google Drive)
- No AWS account or credentials, because the data is public
- No software to install
- Patience and time for the large steps: the download and the Sst extraction each take several minutes


---



<a name="colab-intro"></a>
# Intro to Colab



Welcome to Google Colab! This is the notebook environment we will use to access the data.

</br>
<img src="https://miro.medium.com/max/986/1*pimj8lXWwZnqLs2xVCV2Aw.png" width=300>
</br>
</br>


## What is Colab?
Colab is a browser-based environment - you can develop, run, and share code directly on your browser without having to install any software or libraries on your computer. Additionally, code written and run from colab runs on google's cloud computers, so you aren't limited by your laptop's speed or compute power. 📶

</br>


Colab can be especially helpful for data science tasks, because it gives you a space to write code, generate/view graphs, and organize your thought process.  For those of you who have used Jupyter before, colab acts a simple browser-based Jupyter notebook. We'll be using colab with Python, but colab can be used with a variety of different coding languages.

</br>

A Colab notebook (a file like this one) consists of cells, snippets of code or text that you can create, edit, and move around to keep things organized. These cells can be:

*   **text cells** 🔠
<br>or<br>
*   **code cells** 💻

You can **double click** <img src='https://miro.medium.com/v2/resize:fit:280/0*oa0XcvM99Y5clDsj.png' alt='mouse cursor images' height='25px'>
on a cell in order
to edit it.
</br>

----
## 0.1 Text Cells
<a name="text"></a>



This is a **text cell**. You can **double-click** to edit this cell. Text cells
use markdown syntax. To learn more, see the [markdown
guide](/notebooks/markdown_guide.ipynb).

You can format text cells by **changing** `the` *font*,  adding:
1.   Numbered lists, or
*   Bullet points


You can even add HTML to your text cells to embed images or add other fun effects.


<img src = "https://freesvg.org/img/icon-html.png" width=50>


You can also add math to text cells using [LaTeX](http://www.latex-project.org/)
to be rendered by [MathJax](https://www.mathjax.org). Just place the statement
within a pair of **\$** signs. For example `$\sqrt{3x-1}+(1+x)^2$` becomes
$\sqrt{3x-1}+(1+x)^2.$ <br><br>
To insert a text cell, click on a cell on the page and then click the "+ Text" button at the top. Or scroll to the middle of the notebook page before/after a cell and you should see these options appear: <br>
<img src="https://github.com/tmckim/python-crash-course/blob/main/imgs/ColabAddCodeTextImg.png?raw=1"/>


This will make a text cell directly above or below the cell you are on.


---
## 0.2 Code Cells
<a name="code-cells"></a>

Colab notebooks also consist of **coding cells**, cells that have executable code written in them.

To run a coding cell, either:
1. hover over the cell and click the "play" ▶ arrow that appears on the left upper corner
2. Or click into the cell and then press Ctrl+Enter (or Shift+Return).

You should see a check mark if the cell runs successfully ✔.

Coding cells can look like the following, where you can see raw code and comments (anything after the `#hashtag`). <br>

In [ ]:
parts = 6
# This is a comment! When there is a # before a line in a coding cell,
# python will ignore that line of code, rather than trying to run it.

'''
You can also
write longer
comments
by using
this syntax.
'''

print("Number of sections in this notebook:", parts)

In [ ]:
statement = "Hello, World!"
print(statement)

Additional resources to practice with Colab notebooks and get you started if you want even more before diving into the notebook below:

* [Welcome to Colab](https://colab.research.google.com/notebooks/intro.ipynb)
* [Colab Overview](https://colab.research.google.com/notebooks/basic_features_overview.ipynb)
* [External Data](https://colab.research.google.com/notebooks/io.ipynb)



---



<a name="intro-snrnaseq"></a>
# What is single-nucleus RNA-seq (snRNA-seq)?

**Single-nucleus RNA-seq (snRNA-seq)** measures gene expression in individual isolated nuclei rather than whole cells.

**Single-cell RNA-seq (scRNA-seq)** measures gene expression in individual whole cells, including both cytoplasmic and nuclear transcripts.

**Bulk RNA-seq** measures average gene expression across a population of thousands to millions of cells, losing all cell-type-specific resolution.


Sequencing nuclei (rather than whole cells) works well for frozen post-mortem brain tissue, where intact cells may be less likely to survive extraction relative to nuclei.

The output is one row of gene-expression measurements *per nucleus*. A brain tissue sample like the middle temporal gyrus (MTG) contains many different cell types, including: excitatory neurons, inhibitory interneurons, astrocytes, microglia, and more. Alzheimer's Disease (AD) may affect these cell types very differently.

snRNA-seq data lets us ask cell-type-specific questions: <br>
    - *Which* cell types change gene expression (loss or gain), or disappear as Alzheimer's pathology progresses?

That's the central question this notebook explores, zooming in on one especially vulnerable interneuron type: **somatostatin (SST) neurons**.

---


<a name="intro-dataset"></a>
# About the Dataset

- This dataset comes from the Seattle Alzheimer's Disease Brain Cell Atlas (SEA-AD), described in [Gabitto, Travaglini, et al. (2024, *Nature Neuroscience*)](https://www.nature.com/articles/s41593-024-01774-5).
- It contains single-nucleus RNA-seq (snRNA-seq) data from ~3.4 million nuclei within the middle temporal gyrus (MTG) of 84 donors spanning the full range of Alzheimer's pathology, from no/low pathology to severe disease
- Nuclei are annotated into 139 molecular cell types using a reference taxonomy from the BRAIN Initiative
    - Includes neuronal and non-neuronal populations
- The aim of the study is to identify which brain cell types are most vulnerable to AD and at what stage of disease progression
- Findings revealed an early phase marked by loss of somatostatin-positive inhibitory neurons and a later phase marked by excitatory neuron loss
- In this notebook, we'll work with this snRNA-seq data to explore cell types, gene expression patterns, and how they relate to disease pathology


Video overviews:
- Full webinar (~1.5 hr): https://www.youtube.com/watch?v=lcS3o1N22Bc
- Webinar overview (~7 mins): https://youtu.be/QON_qTo06oM


----

In [ ]:
#@title Allen Institute SEA-AD Webinar Overview (~7 mins) {display-mode: "form"}

from IPython.display import HTML

HTML("""
<iframe width="800" height="450"
        src="https://www.youtube.com/embed/QON_qTo06oM"
        title="YouTube video player"
        frameborder="0"
        referrerpolicy="strict-origin-when-cross-origin"
        allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
        allowfullscreen>
</iframe>
""")

<a name="intro-anndata"></a>
# The AnnData ("Annotated Data") object

Data from this experiment, which contains ~1.4 million nuclei, the expression level of ~36,600 genes, plus donor demographics, QC metrics, and pre-computed coordinates, needs a container that keeps all of that aligned.

The standard for single-cell data in Python is **AnnData** (`adata`), from the [scverse](https://anndata.scverse.org/en/stable/) ecosystem:

<div align="center">
<img src="https://anndata.scverse.org/en/stable/_images/anndata_schema.svg" width="400">
</div>

**General explanation for AnnData**
- `X Matrix`: The central cell-by-feature (e.g., cell-by-gene) expression data matrix
- `Obs` / `Var`: Data frames storing metadata annotations for observations (cells) and variables (genes/features)
- `Obsp` / `Varp`: Pairwise relationships between observations or variables
  - Examples: k-nearest neighbors (KNN) connectivity graphs and cell/gene similarity matrices
- `Obsm` / `Varm`: Multi-dimensional annotations and embeddings
  - Examples: UMAP, t-SNE, and PCA coordinates
- `Layers`: Additional alternative count matrices, such as raw counts versus normalized data
- `Uns`: Unstructured metadata not tied to a specific dimension
  - Examples: color palettes, clustering parameters, or analysis settings
---

**AnnData in the current file we are working with:**
| Piece | Shape | What it holds |
|---|---|---|
| `X` | cells × genes | The expression matrix itself, with the bulk of the data, stored as a sparse matrix |
| `layers['UMIs']` | cells × genes | Raw UMI counts, kept alongside the normalized `X` |
| `obs` | cells × metadata | Per-cell metadata: donor demographics, QC metrics, cell type calls. Rows line up 1:1 with `X` rows |
| `var` | genes × metadata | Per-gene metadata (`_index` = gene symbols, `gene_ids` = e.g. Ensembl IDs). Rows line up 1:1 with `X` columns |
| `obsm` | cells × k | Per-cell multi-dimensional arrays: `X_scVI` (latent embedding), `X_umap` (2D coordinates) |
| `obsp` | cells × cells | Cell-to-cell graphs: `distances` and `connectivities` from the nearest-neighbor graph |
| `uns` | — | Unstructured extras: color palettes, processing parameters, dataset title, etc. |
---

In scRNA-seq data, each row corresponds to a cell with a barcode, and each column corresponds to a gene with a gene id (cells × genes contained in `X`)

`obs` and `var` are the row/column labels for everything else

Once you know a cell's row index, that same index tells you its coordinates in `obsm['X_umap']`, its neighbors in `obsp`, and its expression values in that row of `X`.

**Try it yourself:** Before running anything below, look at the diagram above.
- Which box do you think holds *gene names*?
- Which one holds *cell metadata* like donor age or disease status?

You'll open the real object in [Part 2](#part-2) and can check your answer.

---

<a name="part-1"></a>
# Part 1: Setup & Getting the Data

<a name="logistics"></a>
### Logistics for large files

h5ad files (data is ~34 GB) are too large to load fully into memory, so we use different strategies to access and work with the data:

1. **Backed mode** (`anndata`, `backed = 'r'`): opens the file without loading `.X` into RAM
    - explore `.obs` (cell metadata) and decide what to subset
2. **Chunked extraction**: once specific genes/cells are chosen, read only that slice out of the underlying sparse arrays via `h5py`

Full file is downloaded first, then opened in backed mode





<a name="get-started"></a>
### Getting Started
The beginning process here requires the following steps to be able to access the data. These parts are more advanced, so if you are just learning, it is okay to just get the gist of what is happening (see comments). The notebook has been setup this way so you don't have to worry about data import and size issues. Just running these first cells will be required each time you start (restart) the notebook.




<a name="sec-1-1"></a>
## 1.1 Setup & Import Packages

> **Note:** Results shown here use `scanpy`. Other tools (e.g., Seurat in R) or even different versions/parameters within `scanpy` can give slightly different outputs, since methods may vary by implementation. Exact numbers may differ, but overall biological conclusions should be consistent.

Install & import the packages needed for running this notebook

In [ ]:
# Install required packages
# anndata/scanpy/h5py/seaborn/boto3 may not all be preinstalled
# numpy, pandas, matplotlib, and scipy typically come along as dependencies of anndata/scanpy

!pip install anndata scanpy h5py seaborn boto3 s3fs -q

**Note**: You may see a warning about a `pandas` version. As of the time of testing (08/2026), this notebook will still run without issues.

In [ ]:
import os

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm
import s3fs
import seaborn as sns
from scipy.stats import mannwhitneyu
from IPython.display import display
import threading
import time

# figure defaults: set once here (rather than in a separate later cell) so a
# second sc.set_figure_params() call doesn't reset dpi/facecolor to scanpy defaults
sc.settings.verbosity = 1
sc.set_figure_params(dpi = 100, facecolor = 'white', figsize = (8, 8))

# Colab's local disk (not Google Drive), which is storage tied to this session
# Not saved once the session ends, but doesn't use any of your personal Drive space
DATA_DIR = '/content'
LOCAL_PATH = os.path.join(DATA_DIR, 'SEAAD_MTG_2024_RNAseq.h5ad')

<a name="sec-1-2"></a>
## 1.2 Locate & download the data

The full `final-nuclei` file (~34 GB) is downloaded directly from the public S3 bucket below, with a live progress bar so you can track it. The cell below skips the download automatically if the file already exists at `LOCAL_PATH`.

- This is the default Google temporary working space under `Files` in the left side bar (click on the 📁 folder icon)
- The path is: `content/`

Note: This uses the 2024 data (`/previous_objects` folder)<br>

A recent update was made available as of June 2026, which is in the main folder `sea-ad-single-cell-profiling/MTG/RNAseq/`
<br>
<br>

Running this cell will take a bit (estimated ~7 minutes)


In [ ]:
# Set up an anonymous S3 client (the bucket is public, no AWS credentials needed)
s3 = boto3.client('s3', config = Config(signature_version = UNSIGNED))

bucket = 'sea-ad-single-cell-profiling'
key = 'MTG/RNAseq/previous_objects/SEAAD_MTG_RNAseq_final-nuclei.2024-02-13.h5ad'   # 2024 data is here

os.makedirs(DATA_DIR, exist_ok = True)                                              # Colab's /content already exists, but ok on any machine

print(f"Data will be stored at: {LOCAL_PATH}")

if os.path.exists(LOCAL_PATH):
    print(f"File already exists at {LOCAL_PATH} ({os.path.getsize(LOCAL_PATH) / (1024**3):.2f} GB) - skipping download")
else:
    # Get the file size first, so the progress bar knows the total
    file_size = s3.head_object(Bucket = bucket, Key = key)['ContentLength']         # total size in bytes
    print(f"File size: {file_size / (1024**3):.2f} GB")

    # Download with a live progress bar
    with tqdm(total = file_size, unit = 'B', unit_scale = True, unit_divisor = 1024, desc = os.path.basename(LOCAL_PATH)) as pbar:
        def progress_callback(bytes_transferred):
            pbar.update(bytes_transferred)                                          # advance the bar by however many bytes just came in

        s3.download_file(bucket, key, LOCAL_PATH, Callback = progress_callback)

    print(f"\nDownload complete: {os.path.getsize(LOCAL_PATH) / (1024**3):.2f} GB")

Everything below assumes that file now exists at the `LOCAL_PATH`

- This is the default Google temporary working space under `Files` in the left side bar (click on the 📁 folder icon)
- The path is: `content/`


(This is printed after the above step - see `Data will be stored at:`)


<a name="sec-1-3"></a>
## 1.3 View all data available online associated with SEA-AD

Here is the storage space where you can see what other data single-cell data files that are available: https://sea-ad-single-cell-profiling.s3.amazonaws.com/index.html


Each bucket in the SEA-AD registry on AWS holds a different data modality:
- **sea-ad-single-cell-profiling**: single-cell transcriptomics (snRNAseq) & epigenomics (ATACseq)
- **sea-ad-quantitative-neuropathology**: full-resolution & processed pathology images & quantifications
- **sea-ad-spatial-transcriptomics**: spatial transcriptomics data


More info: https://registry.opendata.aws/allen-sea-ad-atlas/


To see all the data in tables and so text isn't cut off when you want to view the information, set the viewing options in `pandas`

In [ ]:
# change viewing options
pd.set_option('display.max_rows', None)       # show every row instead of truncating the middle
pd.set_option('display.max_colwidth', None)   # show full file names instead of cutting them off with "..."

In [ ]:
# examine specific single cell data
fs = s3fs.S3FileSystem(anon = True)
bucket_path = 'sea-ad-single-cell-profiling/'

# detail = True returns dicts directly (type/name already included)
entries = fs.ls(bucket_path, detail = True)

file_info = []
for e in entries:
    file_info.append({
        'name': e['name'].rstrip('/').split('/')[-1],
        'type': e['type'],                                                      # 'file' or 'directory'
        'full_path': e['name'],
    })

pd.DataFrame(file_info).sort_values('name')

In [ ]:
# See what types of data are available
buckets = [
    'sea-ad-single-cell-profiling',
    'sea-ad-quantitative-neuropathology',
    'sea-ad-spatial-transcriptomics',
]

fs = s3fs.S3FileSystem(anon = True)

all_info = []
for bucket in buckets:
    entries = fs.ls(bucket + '/', detail = True)                                # top-level only
    for e in entries:
        all_info.append({
            'bucket': bucket,
            'name': e['name'].rstrip('/').split('/')[-1],
            'type': e['type'],                                                  # 'file' or 'directory'
            'full_path': e['name'],
        })

pd.DataFrame(all_info).sort_values(['bucket', 'type', 'name'])

As you can see, there is much more data available than what we will work with in this notebook! But once you run through this, you could adapt the code to explore different brain regions.

<mark> There is also a companion notebook on **Accessing the SEA-AD Data in Colab**

- This reviews how to access files from other brain regions
- It can be adapted for other data modalities as well

---
<a name="sec-1-4"></a>
## 1.4 Optional: Where to save your plots

Every plotting cell below calls a small `save_plot(...)` helper (or scanpy's own `save =` option, for a couple of UMAP plots) right before it displays. The cell below controls where those files go.

Pick **ONE** of the two options in it before continuing.

In [ ]:
# Choose where plots get saved. Pick ONE of the two options below.

SAVE_PLOTS = True    # set to False to skip saving plots entirely

# Option A (default): Colab's local disk. Fast, but everything here is
# deleted once the session ends, unless you download it first (sidebar folder icon)
PLOT_DIR = '/content/plots'

# Option B: your own Google Drive, so persists across sessions. To use this
# instead, uncomment the 3 lines below (this will prompt you to authorize access)
# from google.colab import drive
# drive.mount('/content/drive')
# PLOT_DIR = '/content/drive/MyDrive/SEAAD_MTG_RNAseq_plots'

os.makedirs(PLOT_DIR, exist_ok = True)
sc.settings.figdir = PLOT_DIR                                                   # scanpy's own plots (the two UMAPs using save) go here too
print(f"Plots will be saved to: {PLOT_DIR}")


def save_plot(filename):
    """Save the current matplotlib figure to PLOT_DIR, if SAVE_PLOTS is enabled.
    Call this right before plt.show() in any plotting cell."""
    if SAVE_PLOTS:
        path = os.path.join(PLOT_DIR, filename)
        plt.savefig(path, dpi = 150, bbox_inches = 'tight')
        print(f"Saved: {path}")


---
<a name="part-2"></a>
# Part 2: Opening the Data & Exploring Variables



<a name="sec-2-1"></a>
## 2.1 Open the file (backed mode) & inspect structure

In AnnData, cells = observations (`obs`) & genes = variables (`var`)
- Backed mode (`backed='r'`) is designed to give quick access to metadata without loading the large expression matrix (`.X`) into memory
- `ad.read_h5ad(LOCAL_PATH, backed='r')` can crash on a file this size (~34 GB) in memory-constrained environments like Colab
- Since parts of the notebook only needs `obs`, the precomputed UMAP layout, and the Subclass color palette (we don't use `.X` due to size), the cell below reads just those pieces directly via `h5py` instead, then wraps them in a lightweight AnnData object with the same interface
<br>
<br>

Running this cell could take ~1 minute or so

In [ ]:
# Access data

# `obs` columns aren't all categoricals, some use other encodings
# (e.g. nullable integers/booleans)
try:
    from anndata.io import read_elem                                            # newer anndata versions
except ImportError:
    from anndata.experimental import read_elem                                  # older anndata versions

with h5py.File(LOCAL_PATH, 'r') as f:
    obs_full = read_elem(f['obs'])                                              # full obs DataFrame
    n_vars = f['var'][f['var'].attrs['_index']].shape[0]                        # number of genes, from the var index length
    umap_coords = read_elem(f['obsm']['X_umap'])                                # precomputed 2D UMAP layout
    subclass_colors = read_elem(f['uns']['Subclass_colors'])                    # hex color palette used for Subclass

adata_meta = ad.AnnData(
    obs = obs_full,
    obsm = {'X_umap': umap_coords},
    uns = {'Subclass_colors': subclass_colors},
    shape = (obs_full.shape[0], n_vars),                                        # no X is loaded, shape is passed explicitly instead
)

The results should be: 1,378,211 cells x 36,601 genes

In [ ]:
# Print shape (cells x genes)
print(f"Dataset shape: {adata_meta.n_obs} cells x {adata_meta.n_vars} genes")

In [ ]:
# Print column names in `obs`
print(f"\n--- Cell metadata columns (obs) ---")
for col in adata_meta.obs.columns:
    print(col)

----

Info regarding data columns can be found in the supplemental material associated with the paper here: https://www.nature.com/articles/s41593-024-01774-5#Sec28


Some info also found in the `README.md` file with the data in the AWS bucket: https://sea-ad-single-cell-profiling.s3.amazonaws.com/README.md


---

The file extension of the data file is `.h5ad`. <br>
This is a specialized HDF5-based format used in single-cell bioinformatics. Files are organized like a filesystem: groups (like folders) contain datasets (like files) or other groups

Here we loop over this structure to view the organization and what is contained in each

In [ ]:
# View the data structure using h5py
with h5py.File(LOCAL_PATH, 'r') as f:                                           # open the h5ad file directly in read-only mode
    for name in f.keys():                                                       # loop over the top-level keys (e.g. X, obs, var, obsm, layers, etc.)
        item = f[name]                                                          # get the object stored at this key
        if isinstance(item, h5py.Group):                                        # groups are like folders — they contain nested datasets/subgroups
            print(f"{name}/  (group)")                                          # print the group name with a trailing slash
            for sub_name in item.keys():                                        # loop over the items nested inside this group
                print(f"    {sub_name}")                                        # print each nested item name, indented
        else:                                                                   # if it's not a group, it's a dataset (an actual array of data)
            print(f"{name}  shape = {item.shape}, dtype = {item.dtype}")        # if not a group, it's a dataset and can print shape and datatype

These are the pieces from the AnnData diagram above: `X`, `obs`, `var`, `obsm`, `obsp`, `uns`, and `layers` are top-level keys in the raw HDF5 file

<a name="sec-2-2"></a>
## 2.2 Explore cell metadata (`obs`)

In [ ]:
# Review the first 5 rows of the obs dataframe
adata_meta.obs.head()                                                           # head() shows first 5 rows; put in number to see more: head(10)

<a name="sec-2-2-1"></a>

### 2.2.1 Subclass and Supertype columns

The `Subclass` labels weren't generated by clustering this dataset from scratch. Nuclei were mapped onto an existing reference cell type taxonomy built from prior BRAIN Initiative Cell Census Network (BICCN) studies of the MTG. There is a hierarchy of 3 broad classes (excitatory neurons, inhibitory neurons, non-neuronal cells), 24 subclasses (e.g., L2/3 IT excitatory neurons), and 125 finer-grained "supertypes" were defined. SEA-AD nuclei were assigned to this taxonomy using a hierarchical probabilistic Bayesian mapping approach, and 14 additional disease-associated non-neuronal states were added, for a total of 139 supertypes.

`Subclass` reflects this second level of the hierarchy, which is a broader grouping than the full supertype-level annotation.


Let's start to examine the data by looking at the `Subclass` column categories and get counts of cells

In [ ]:
# Get cell counts by subclass
counts = adata_meta.obs['Subclass'].value_counts().sort_index()                 # sort by count and then categorical order
print("Cell counts by subclass:")                                               # text to describe what is printed
print(counts)                                                                   # print the counts
print(f"\nNumber of subclasses: {len(counts)}")                                 # check how many subclass categories there are

`Subclass` is a broad category, and we can narrow down to be more specific within this category using the `Supertype` column categories to get counts of cells

In [ ]:
# Get cell counts by supertype
counts_st = adata_meta.obs['Supertype'].value_counts().sort_index()             # sort by count and then categorical order
print("Cell counts by supertype:")                                              # text to describe what is printed
print(counts_st.to_string())                                                    # print the counts and enable to see more (all) categories than default displayed
print(f"\nNumber of supertypes: {len(counts_st)}")                              # check how many supertype categories there are

---
What we see is that there are more `Supertypes` (subcategories) within each overall `Subclass`

Other `obs` columns worth knowing about:
- `Class`: the broadest grouping: neuronal (split into GABAergic/glutamatergic) vs. non-neuronal
- `Supertype`: above is the *expanded* version, which is the finest-grained cell type info in this dataset
  - You may have also noticed a column called `Supertype (non-expanded)`

---


<a name="sec-2-2-2"></a>

### 2.2.2 Alzheimer's Disease (AD) Pathology Data
We can also explore columns related to AD pathology. The aim of SEA-AD is to link *cellular* changes to *disease progression*. Donor data includes standard neuropathological measures alongside their snRNA-seq data. Scores like Braak stage, CERAD score, and Thal phase are established clinical/pathological staging systems for AD-related brain changes (tangles, plaques, and amyloid spread, respectively).

Newly developed in this paper, the Continuous Pseudo-progression Score (CPS) is a metric that places each donor along a continuous trajectory of disease severity rather than discrete stages.


Other columns capture related but distinct information such as: cognitive status, APOE genotype (a major genetic risk factor for AD), co-occurring pathologies like Lewy body disease and cerebral amyloid angiopathy (CAA), and whether a donor served as a neurotypical reference.


Exploring these columns lets us connect cell-type composition and gene expression patterns to where each donor falls on the disease spectrum, which is the basis for identifying which cell types are vulnerable and when.

**Note**: For these measures, if you see data labeled as `Reference`, these samples were from young donors from the BRAIN initiative that were mapped to SEA-AD data. They are not considered part of the SEA-AD cohort.
- There was also a column in `obs` above that would label these as `Neurotypical reference == True`

**Note**: If you see `Reference` in these columns, those cells come from a separate group of neurotypical reference donors, not the 84 SEA-AD donors. They were included as a reference for cell-type classification. The same cells are marked `Neurotypical reference == True` in `obs`.

- More info here: https://community.brain-map.org/t/sea-ad-donors-and-samples/4669

---

In [ ]:
# Columns known to hold AD-pathology relevance (stored in obs)
ad_cols = [                                                                     # create a list to store col names
    'APOE Genotype',
    'Braak',
    'CERAD score',
    'Cognitive Status',
    'Continuous Pseudo-progression Score',
    'Highest Lewy Body Disease',
    'Neurotypical reference',
    'Overall AD neuropathological Change',
    'Overall CAA Score',
    'Severely Affected Donor',
    'Thal',
]
print("AD-related metadata columns:", ad_cols)                                  # text to print and tell us what we are viewing

for col in ad_cols:                                                             # loop through the cols in the list above
    print()                                                                     # blank line between columns (space to better view output)
    print(adata_meta.obs[col].value_counts().sort_index())                      # for each category, print count of cells in order defined from data

**Try it yourself:** Pick one `obs` column we didn't explore above already (see the full column list printed in [Section 2.1](#sec-2-1)) and run `.value_counts()` on it.

Anything surprising?

In [ ]:
## Your code here


---
<a name="sec-2-3"></a>
## 2.3 Quality Control (QC) & Uniform Manifold Approximation and Projection (UMAP)

<a name="sec-2-3-qc"></a>
### 2.3.1 Quality Control (QC) Checks

Before analyzing single-nucleus RNA-seq data, it's standard practice to look at a few basic QC metrics per nucleus:

- **Genes detected per nucleus**: how many distinct genes were detected
    - Very low values often indicate empty droplets or poor-quality nuclei
    - Unusually high values can indicate doublets (two nuclei captured together)
- **Total Unique Molecular Identifier (UMI) counts per nucleus**: total transcript molecules detected
    - Similar interpretation to genes detected; very low or very high counts are flagged for removal
- **Nuclei per subclass/cell type**:
    - Sanity check that no cell type is over- or under-represented
    - Which would hint at batch effects or annotation issues

<u>Note on this dataset</u>: SEA-AD data has already been processed and QC-filtered, so we're not using these plots to set filtering thresholds or remove any cells. We're just confirming the data looks clean, which is good practice even on a dataset you trust.
<br>
<br>

**If you're starting from a raw/unfiltered dataset**: these same plots are typically the first step of analysis, used to choose thresholds for filtering out low-quality nuclei (or cells) before anything else (clustering, differential expression, etc.). Some resources for how to approach that:

- [Single-cell best practices - Quality Control](https://www.sc-best-practices.org/preprocessing_visualization/quality_control.html): a thorough, actively maintained guide covering QC metrics, thresholding strategies, and common pitfalls
- [Scanpy's preprocessing & clustering tutorial](https://scanpy.readthedocs.io/en/stable/tutorials/basics/clustering.html): walks through QC and filtering using the exact `scanpy` functions (`sc.pp.calculate_qc_metrics`, `sc.pl.violin`, etc.) that would apply here
- [Luecken & Theis, *"Current best practices in single-cell RNA-seq analysis: a tutorial"*, Molecular Systems Biology (2019)](https://link.springer.com/article/10.15252/msb.20188746): paper covering QC and the broader analysis workflow



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))                                 # 3 subplots; 1 row, 3 cols; size in (width, height)

# 1: hist of genes detected
axes[0].hist(adata_meta.obs['Genes detected'], bins = 50, color = 'steelblue')
axes[0].set_xlabel('Genes detected per nuclei')                                 # label x-axis
axes[0].set_title('Gene count distribution')                                    # title
axes[0].tick_params(axis = 'both', labelsize = 8)                               # ticks on both axes & set font size

# 2: hist of UMIs
axes[1].hist(adata_meta.obs['Number of UMIs'], bins = 50, color = 'darkorange')
axes[1].set_xlabel('Total UMI counts per nuclei')
axes[1].set_title('UMI count distribution')
axes[1].tick_params(axis = 'both', labelsize = 8)

# 3: bar plot of cell counts per subclass
adata_meta.obs['Subclass'].value_counts().plot(kind = 'bar', ax = axes[2], color = 'seagreen')
axes[2].set_title('Nuclei per subclass')
plt.setp(axes[2].get_xticklabels(), rotation = 45, ha = 'right', fontsize = 8)
axes[2].tick_params(axis = 'y', labelsize = 8)

plt.tight_layout()
save_plot('01_qc_metrics.png')
plt.show()

<a name="sec-2-3-umap"></a>
### 2.3.2 UMAP visualization

Since we are working with published data, the AnnData object has built in information we can easily use to replicate the UMAP visualization shown in [Figure 3A (middle)](https://www.nature.com/articles/s41593-024-01774-5/figures/3)

**Note**: t-SNE is another common dimensionality reduction technique used for visualizing single-nuclei (or single-cell) data, but we won't cover it here since the dataset provides pre-computed UMAP coordinates.

Which one is "better" is a matter of ongoing debate in the field. In practice, both are approximations that can distort the underlying high-dimensional data, so the choice often comes down to convention within a subfield or dataset compatibility rather than one being definitively correct.

In [ ]:
# Use scanpy's (sc) plotting (pl) function for UMAPs (umap)
sc.pl.umap(adata_meta,                                                          # anndata obj containing precomputed UMAP layout in: adata_meta.obsm['X_umap']
           color = 'Subclass',                                                  # colors based on Subclass in adata_meta.obs & using palette from adata_meta.uns['Subclass_colors']
           title = 'MTG RNAseq - Subclass',                                     # title
           show = False)                                                        # don't display yet, save_plot() below needs the figure to still be open
save_plot('02_umap_subclass.png')
plt.show()

**Try it yourself**: Customize the UMAP plot

Using the [`sc.pl.umap` documentation](https://scanpy.readthedocs.io/en/stable/api/generated/scanpy.pl.umap.html#scanpy.pl.umap), modify the plot below to:

1. Put the cluster labels **directly on the plot** instead of in a side legend
2. Make the dots larger and more opaque so overlapping clusters are easier to distinguish
3. Remove the surrounding axis box/ticks, since UMAP axes don't have meaningful units
4. **Bonus:** color the plot by `'Supertype'` instead of `'Subclass'` and compare how the clustering looks at that finer resolution

Hint: all of these are keyword arguments to `sc.pl.umap()` — you don't need to change anything else about the code.

In [ ]:
## Your code here


<a name="sec-2-4"></a>
## 2.4 Cell subclass composition by AD pathology stage

So far we've looked at cell types across the full dataset. Now we'll start looking at how those cell types relate to Alzheimer's disease pathology, beginning with a simple question: <br>
**Does the mix of cell subclasses change as AD pathology severity increases?**

A stacked bar chart is a straightforward way to see this: each bar represents one ADNC (<u>AD</u> <u>N</u>europathological <u>C</u>hange) category, and the colored segments show what fraction of cells in that category belong to each subclass.

This makes relative composition easy to compare across groups, and sets up more detailed AD-focused analysis later in the notebook.



In [ ]:
col_of_interest = 'Overall AD neuropathological Change'                         # colname - AD severity variable

# build a cross-tab table with pandas (pd)
proportions = pd.crosstab(adata_meta.obs[col_of_interest],                      # rows are AD severity categories
                          adata_meta.obs['Subclass'],                           # cols are cell subclass
                          normalize = 'index')                                  # convert each row from raw counts into props that sum to 1

fig, ax = plt.subplots(figsize=(8, 6))                                          # fig setup/dims: single figure, 8x6 inches
# plotting code
proportions.plot(                                                               # plot the proportions df
    kind = 'bar',                                                               # pandas (matplotlib in background)
    stacked = True,                                                             # stacked bar
    ax = ax,                                                                    # draw on the ax setup above
    color = adata_meta.uns['Subclass_colors'],                                  # palette from anndata for subclass colors
    width = 0.8,                                                                # bar width (leaves gaps between each)
)
# set plot parameters
ax.set_xlabel(col_of_interest)                                                  # define x-axis label based on col chosen above
ax.set_ylabel('Proportion of cells')                                            # define y-axis label
ax.set_title(f'Subclass composition by {col_of_interest}')                      # title based on col chosen above
ax.legend(bbox_to_anchor = (1.02, 1), loc = 'upper left', fontsize = 8)         # location and size of legend text
ax.tick_params(axis = 'x', rotation = 45)                                       # rotate x-axis ticks 45 degrees (non-overlapping)
ax.grid(False)                                                                  # remove background grid
plt.tight_layout()                                                              # adjust spacing to all fit without cutoff
save_plot('03_subclass_composition_by_adnc.png')
plt.show()                                                                      # show the figure


In [ ]:
# Show the raw numbers for a specific subclass (e.g., Sst) across the AD severity categories
print(proportions['Sst'])

<a name="sec-2-4-interp"></a>
### Interpretation: subclass composition plot

**Note**: This plot pools all nuclei together before computing proportions, so it is a nucleus-weighted average rather than a donor-weighted one. Donors who happened to yield more nuclei (due to tissue quality, number of nuclei sampled, etc.) count more toward each bar than donors who yielded fewer. If nuclei counts per donor aren't roughly even, a few high-yield donors could be driving or masking the pattern in a given group. This could also be true if the number of donors per ADNC categories is unbalanced. A donor-level version could be computed instead, where each donor's own proportions are averaged within each ADNC group, so every donor counts equally rather than every nucleus.

For simplicity, we don't include a donor-level version here. This is doable with code, and anyone wanting to take this a step further can test it out and compare the two proportion plots.

<a name="sec-2-4-reference"></a>
### Note on ADNC "Reference" Category

There should only be 5 donors with `ADNC == 'Reference'`. These are samples from young donors from the BRAIN initiative that were mapped to SEA-AD data. They are not considered part of the SEA-AD cohort.

More info here: https://community.brain-map.org/t/sea-ad-donors-and-samples/4669

In [ ]:
# confirm how many unique donors are labeled 'Reference'
ref_donors = adata_meta.obs.loc[adata_meta.obs['Overall AD neuropathological Change'] == 'Reference', 'Donor ID']  # get the Donor ID for every cell whose 'Overall AD neuropathological Change' is 'Reference'
print(ref_donors.nunique())                                                                                        # count how many distinct donors are in that filtered list

**Try it yourself:** Pick a different subclass you noticed in the proportion plot above. Using the code example above (you can copy and paste!), print the proportion values so you can see the numbers for this subclass. How does its pattern compare across AD severity? What about to Sst?

In [ ]:
## Your code here


---
<a name="sec-2-4-paper"></a>
### Connecting back to the paper

At the subclass level, Sst's share of cells looks fairly flat across ADNC severity groups because no dramatic decline jumps out. That's not necessarily inconsistent with [Gabitto, Travaglini, et al. (2024)](https://www.nature.com/articles/s41593-024-01774-5). The paper's key finding is that only *specific* Sst supertypes are selectively vulnerable early in disease progression. Aggregating everything up to the subclass level averages vulnerable and stable supertypes together, which can mask this kind of effect.

The paper also takes a more specific approach to timing and progression of the underlying neurobiological changes in AD than the categorical columns we used above (`Overall AD neuropathological Change`). The researchers built a **Continuous Pseudo-progression Score (CPS)**: a Bayesian model integrated multiple quantitative neuropathology measurements per donor into a single continuous 0-1 score representing where each donor falls along the disease trajectory. CPS captures much finer-grained timing than a categorical bin, and it's how they were able to detect that vulnerable Sst supertypes decline early and continuously with CPS, while other cell types (like L2/3 IT excitatory neurons and Pvalb interneurons) only decline sharply at high CPS. This is part of why the pattern looks flat in our simpler overall subclass and one categorical variable for AD proportion plot, but is detectable with the finer-resolution approach used in the paper.


---


**Try it yourself:** Since vulnerability isn't uniform across all Sst cells, breaking the composition down by individual Sst supertypes (rather than the Sst subclass as a whole) may be more likely to surface the early decline the paper describes.

Create a new proportion plot based on the code above, but swapping out `Subclass` for `Supertype`

In [ ]:
## Your code here


**Try it yourself:** The stacked bar chart above split cells by `Overall AD neuropathological Change`. Try substituting a different AD-pathology column from [Section 2.2.2](#sec-2-2-2) (e.g. `'Braak'` or `'Cognitive Status'`) and see whether the plot changes.

In [ ]:
# if you've run this notebook in order, this should be available to you

print("AD-related metadata columns:\n")
for col in ad_cols:
    print(col)

In [ ]:
## Your code here


---

<a name="part-3"></a>
# Part 3: Zooming In -  Somatostatin (SST) Interneurons


<a name="sec-3-1"></a>
## 3.1 Why SST?

One of the findings from the SEA-AD paper ([Gabitto, Travaglini, et al. (2024)](https://www.nature.com/articles/s41593-024-01774-5)) highlights **somatostatin (SST)** interneurons. SST is a neuropeptide, and some research has shown that a subset of SST cells decrease early in Alzheimer's disease, before many other pathological changes are detectable.


That makes SST interneurons a place to practice a full single-nucleus RNAseq data workflow: subset to one cell type, examine gene expression, and later compare a "vulnerable" subset of SST supertypes against the rest.

<a name="sec-3-2"></a>
## 3.2 Obtain SST subset

Instead of working with the full dataset, we'll create a version with all other cell types filtered out, keeping only Sst cells. This makes the rest of the notebook faster to run and easier to reason about

Running this cell will take a bit (estimated ~10 minutes)

In [ ]:
# subset to Sst cells, using the Subclass column already on `adata_meta`
sst_mask = (adata_meta.obs['Subclass'] == 'Sst').values
print(f"{sst_mask.sum()} Sst cells")

# Read only the Sst rows out of the sparse X matrix, in chunks
import scipy.sparse as sp

CHUNK_SIZE = 50_000

with h5py.File(LOCAL_PATH, 'r') as f:
    indptr = f['X']['indptr'][:]                                                # row boundaries into data/indices, for every cell
    n_obs_full, n_genes = f['X'].attrs['shape']                                 # full matrix shape, stored as an attribute
    data_ds = f['X']['data']
    indices_ds = f['X']['indices']
    var_names = f['var'][f['var'].attrs['_index']][:].astype(str)

    row_blocks = []
    for start in tqdm(range(0, n_obs_full, CHUNK_SIZE), desc = 'Extracting Sst rows'):
        end = min(start + CHUNK_SIZE, n_obs_full)
        block_mask = sst_mask[start:end]
        if not block_mask.any():
            continue                                                            # no Sst cells in this chunk, skip it

        row_start, row_end = indptr[start], indptr[end]
        chunk_csr = sp.csr_matrix(
            (data_ds[row_start:row_end], indices_ds[row_start:row_end], indptr[start:end + 1] - row_start),
            shape = (end - start, n_genes),
        )
        row_blocks.append(chunk_csr[block_mask])                                # keep only the Sst rows in this chunk

    sst_X = sp.vstack(row_blocks).tocsr()

print(f"Extracted X: {sst_X.shape}")

In [ ]:
# build the Sst-only AnnData object from the extracted matrix + matching obs rows
# UMAP coords are pulled in here too, same sst_mask, same source object

sst = ad.AnnData(
    X = sst_X,
    obs = adata_meta.obs[sst_mask].copy(),
    var = pd.DataFrame(index = var_names),
    obsm = {'X_umap': adata_meta.obsm['X_umap'][sst_mask]},
)
print(sst)

In [ ]:
# clean up column names by removing '/'
# pandas filtering (adata_meta.obs[sst_mask]) keeps every original
# category label, even ones with zero cells in this subset, which breaks
# scanpy plots later that try to build a row for every category

cols_slash = []
for c in sst.obs.columns:
    if '/' in c:
        cols_slash.append(c)

rename_map = {}
for c in cols_slash:
    rename_map[c] = c.replace('/', '_or_')

sst.obs = sst.obs.rename(columns = rename_map)
print(f"Renamed columns: {rename_map}")

for col in sst.obs.select_dtypes(['category']).columns:
    sst.obs[col] = sst.obs[col].cat.remove_unused_categories()

**Optional**: Save the SST subset data to /content folder. This is temporary storage associated with each session. If you want a copy of the data file, download  from the sidebar (folder icon) to store locally wherever you prefer.

Running this cell will take a bit (~ 2 minutes) and the file is ~ 2 GB

In [ ]:
# @title Optional: Saving SST data subset file
def save_with_progress(adata, path, poll_interval = 1.0):
    """Write `adata` to `path` in a background thread, while polling the file's
    growing size on disk to drive a live progress readout in the main thread."""

    def _write():
        adata.write(path)

    thread = threading.Thread(target = _write)
    thread.start()

    with tqdm(desc = 'Writing to disk', unit = 'MB') as pbar:
        last_size_mb = 0
        while thread.is_alive():
            if os.path.exists(path):
                size_mb = os.path.getsize(path) / (1024**2)
                pbar.update(size_mb - last_size_mb)                             # advance by however many MB were written since the last check
                last_size_mb = size_mb
            time.sleep(poll_interval)

        thread.join()
        if os.path.exists(path):                                                # catch any final bytes written after the loop's last check
            size_mb = os.path.getsize(path) / (1024**2)
            pbar.update(size_mb - last_size_mb)

    print(f"Sst subset saved to {path} ({os.path.getsize(path) / (1024**3):.2f} GB)")


SST_SAVE_PATH = os.path.join(DATA_DIR, 'SEAAD_MTG_SST_Subset_2024.h5ad')
save_with_progress(sst, SST_SAVE_PATH)

<a name="sec-3-2-vtypes"></a>
### 3.2.1 Vulnerable types

A subset of Sst supertypes were flagged as especially **vulnerable**. Their abundance declines with AD pathology more than other supertypes. We'll build on this `vtypes` list throughout the rest of [Part 3](#part-3) and [Part 4](#part-4).

This information was obtained from the [SEA-AD_2024 github repo](https://github.com/AllenInstitute/SEA-AD_2024/blob/126b3cf77095c4e0ac851ff579895b0e02c2d790/Spatial%20Transcriptomics/06_Cell_proportions.ipynb#L104)

In [ ]:
# list of vulnerable supertypes
vtypes = [
    'Lamp5_3', 'Lamp5_5',
    'Sncg_1', 'Sncg_2', 'Sncg_8',
    'Vip_1', 'Vip_2', 'Vip_11', 'Vip_12', 'Vip_13',
    'Sst_2', 'Sst_3', 'Sst_11', 'Sst_19', 'Sst_20', 'Sst_22', 'Sst_23', 'Sst_25',
    'Pvalb_2', 'Pvalb_3', 'Pvalb_5', 'Pvalb_6', 'Pvalb_8', 'Pvalb_14', 'Pvalb_15',
    'L2/3 IT_1', 'L2/3 IT_3', 'L2/3 IT_5', 'L2/3 IT_6', 'L2/3 IT_7', 'L2/3 IT_8',
    'L2/3 IT_10', 'L2/3 IT_12', 'L2/3 IT_13',
    'Astro_2',
    'OPC_2',
    'Oligo_2',
    'Micro-PVM_3-SEAAD',
]

In [ ]:
# find all vulnerable supertypes that belong to the Sst subclass and make a list of just those
vulnerable_sst = []                                                             # empty list to collect matching supertype names
for s in vtypes:                                                                # loop through each vulnerable supertype in vtypes
    if s.startswith('Sst_'):                                                    # keep only supertypes whose name starts with "Sst_" (i.e. belong to the Sst subclass)
        vulnerable_sst.append(s)                                                # add matching supertype to the list

# count and print them
print(f"{len(vulnerable_sst)} vulnerable Sst supertypes: {vulnerable_sst}")     # print how many were found and their names

<a name="sec-3-3"></a>
## 3.3 Check expectations - SST expression (Dot plot)

If these cells have been labeled SST cells, we will conduct a sanity check to see that they express Sst. We will also check other marker genes from previous work and a reference MTG atlas in [Hodge et al. (2019)](https://www.nature.com/articles/s41586-019-1506-7) - [Figure 1c](https://www.nature.com/articles/s41586-019-1506-7/figures/1).

We will use:
- SST: the gene itself
- GAD1: pan-GABAergic/inhibitory marker
- LHX6: found to be in these cells in past work
- ADARB2: found to be lower in these cells in past work
- SLC17A7: a pan-excitatory marker, included as a negative control; should be near-absent in Sst cells

In [ ]:
# Create a list of markers and check they are in the dataset
markers = ['SST', 'GAD1', 'LHX6', 'ADARB2', 'SLC17A7']


# Keep only the marker genes that actually exist in this dataset
markers_found = []                                                              # empty list to collect markers that exist in the data
for m in markers:                                                               # loop through each marker gene we want to check
    if m in sst.var_names:                                                      # check if this gene is present in the dataset's gene list
        markers_found.append(m)                                                 # if yes, add it to our list

markers = markers_found                                                         # replace markers with the filtered list
print(f"Markers found in dataset: {markers}")

In [ ]:
# Dot plot - per group (e.g. Subclass), shows the fraction of cells expressing each gene (dot size)
# and the average expression level (dot color)
sc.pl.dotplot(
    sst,                                                                        # our Sst-subset AnnData object
    var_names = markers,                                                        # genes to plot
    groupby = 'Supertype',                                                      # group cells by 'Supertype' for finer resolution
    standard_scale = 'var',                                                     # scale expression per gene (0-1) so genes are comparable on the same color scale
    title = 'Marker expression across Sst cells'
)
save_plot('04_dotplot_markers_by_supertype.png')
plt.show()                                                                      # display the plot

<a name="sec-3-3-interp-dotplot"></a>
### Interpretation: marker dot plot

Here we see the fraction of nuclei expressing each gene (dot size) and the average expression level (dot color). Warmer colors indicate higher relative mean expression (scaled per gene) and larger dots indicate a larger fraction of nuclei expressing the gene of interest.

GAD1 and LHX6 are consistently high across nearly all Sst supertypes, matching expectations for pan-GABAergic and MGE-lineage markers. Our negative control (SLC17A7) is uniformly low, as expected.

SST itself is more variable: some supertypes (e.g., Sst_22, Sst_11, Sst_20) show strong SST expression, while others (e.g., Sst_1, Sst_4, Sst_9) show weaker expression. You might expect Sst expression to be high across all, but supertypes are defined by overall transcriptomic similarity across many genes, not by SST expression alone. Some Sst-lineage subtypes can express the SST gene itself more weakly than others. Next, we'll check whether marker expression shifts with AD severity instead of supertype.

<a name="sec-3-4-validating"></a>
### 3.4 Examining supertype identity with data-driven markers

So far we've relied on a small set of marker genes we picked ourselves. We can also use a data-driven approach to reveal top marker genes for each Sst supertype, rather than genes chosen in advance. This helps us to see whether chosen markers could reflect what most distinguishes these supertypes from one another, or whether there are other genes that separate them even more strongly. Here we will illustrate how that works.

Running this cell will take a bit (estimated ~6-8 minutes)

In [ ]:
# Find cluster markers
# Runs a statistical test (Wilcoxon rank-sum by default) to identify genes that are
# significantly enriched in each group relative to all other groups
sc.tl.rank_genes_groups(
    sst,                                                                        # our Sst-subset AnnData object
    groupby = 'Supertype',                                                      # find markers per Supertype (finer resolution than Subclass)
    method = 'wilcoxon',                                                        # statistical test used to rank genes
    key_added = 'rank_genes_supertype'                                          # store results under this key so they don't overwrite other results
)

# Show the top marker genes for each Supertype group
top_markers_df = sc.get.rank_genes_groups_df(sst, group = None, key = 'rank_genes_supertype').groupby('group').head(5)
display(top_markers_df)

# Size the figure based on how many groups (rows) and unique genes (columns)
# actually end up in the plot,
n_groups = sst.obs['Supertype'].nunique()
n_unique_genes = top_markers_df['names'].nunique()

# Plot the top marker genes per group as a dot plot to see which genes show up
# among the top markers for Sst supertypes
sc.pl.rank_genes_groups_dotplot(
    sst,                                                                        # our Sst-subset AnnData object
    key = 'rank_genes_supertype',                                               # use the results computed above
    n_genes = 5,                                                                # show top 5 marker genes per group
    groupby = 'Supertype',                                                      # how to group plot - sst subtypes
    dendrogram = False,                                                         # skip the automatic dendrogram calculation; avoids extra PCA computation
    figsize = (max(10, n_unique_genes * 0.35) + 3,
               max(6, n_groups * 0.35)),                                        # adjust size manually here to override default; otherwise squished plot

)

save_plot('05_dotplot_datadriven_markers.png')
plt.show()                                                                      # display the plot

<a name="sec-3-3-interp-datadriven"></a>
#### Interpretation: data-driven marker dot plot

Each Sst supertype has its own distinct set of top marker genes, which you can look up to find out more about each. Our original genes were useful for confirming these cells belong to the Sst class overall, since they're broadly shared across nearly all supertypes. But [`rank_genes_groups`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.rank_genes_groups.html) looks for genes that stand out in one specific supertype versus the rest, which is a different question. Since our original genes are shared broadly rather than unique to any one supertype, it makes sense they don't show up here.


Next, we'll check whether marker expression shifts with AD severity instead of supertype.

In [ ]:
# Dot plot - our selected marker list above, but grouped by ADNC status instead
# of cell type to check whether marker expression shifts with disease severity
sc.pl.dotplot(
    sst,                                                                        # our Sst-subset AnnData object
    var_names = markers,                                                        # same marker genes as above
    groupby = 'Overall AD neuropathological Change',                            # group cells by ADNC status instead of cell type
    standard_scale = 'var',                                                     # scale expression per gene (0-1) so genes are comparable on the same color scale
    title = 'Marker expression across Sst cells by ADNC status'
)
save_plot('06_dotplot_markers_by_adnc.png')
plt.show()                                                                      # display the plot

<a name="sec-3-3-interp-adnc"></a>
#### Interpretation: markers by ADNC status

Marker expression looks noticeably different in the Reference group compared to every other ADNC category, but it doesn't decline gradually with disease severity the way we might expect. This points to differences between the Reference donors and the SEA-AD cohort, rather than a true disease effect. As a reminder, these Reference donors were not part of the SEA-AD cohort (they are a neurotypical reference from another dataset). This is a good reminder to double check and understand what is in the dataset before drawing biological conclusions from a plot like this.

**Try it yourself:** To visualize how different genes might be more highly expressed in different Sst subtypes, create another dotplot by adding these 3 additional genes to the `markers` list: TH, NPY, FRZB. What differences do you see across Sst subtypes?

In [ ]:
## Your code here



<a name="sec-3-5"></a>
## 3.5 Sst cluster UMAP visualization

We will visualize the Sst data in a UMAP to remind ourselves what the cluster looked like, as well as overlay the supertype labels for the Sst subtypes. We have all of that information available to us in the Sst subset data.

In [ ]:
# UMAP colored by Supertype cluster labels for Sst

sc.pl.umap(
    sst,                                                                        # our Sst-subset AnnData object
    color = 'Supertype',                                                        # color cells by Supertype cluster identity
    palette = 'tab20',                                                          # use the tab20 color palette
    legend_loc = 'on data',                                                     # draw category labels directly on the clusters
    legend_fontsize = 10,                                                       # shrink label text so it doesn't overlap too much with dense clusters
    legend_fontweight = 'bold',                                                 # bold labels so they're readable against the colored dots
    title = 'Sst Supertype'
)
save_plot('07_umap_sst_supertype.png')
plt.show()                                                                      # display the plot

Next, we'll add our marker genes as their own panels alongside the Supertype UMAP, so we can see where each gene is expressed relative to cluster identity. We'll also highlight the vulnerable Sst supertypes (identified earlier from the paper) in red on the Supertype panel, so it's easy to see whether marker expression looks any different in those specific clusters before we move on to the formal statistical comparison across AD severity.

In [ ]:
# UMAP colored by gene expression + cluster labels
plot_vars = ['Supertype'] + markers                                             # add cluster identity as its own panel alongside the marker genes

# build a color list matching the category order scanpy uses for 'Supertype'
supertype_categories = sst.obs['Supertype'].cat.categories.tolist()             # category order as stored in the data

# build one color per category, in the same order as supertype_categories
supertype_colors = []                                                           # red for vulnerable supertypes, light grey for everything else
for s in supertype_categories:
    if s in vtypes:
        supertype_colors.append('red')
    else:
        supertype_colors.append('lightgrey')

# build one title per panel: first panel is the Supertype UMAP,
# followed by one panel per gene showing its expression
title = ['Supertype (vulnerable = red)']
for m in markers:
    title.append(f'{m} expression')

sc.pl.umap(
    sst,                                                                        # our Sst-subset AnnData object
    color = plot_vars,                                                          # color panels: first by cluster label, then by each marker gene's expression
    palette = supertype_colors,                                                 # custom palette, only applies to the categorical 'Supertype' panel
    ncols = 2,                                                                  # arrange the panels in a 2-column grid
    cmap = 'viridis',                                                           # colormap for expression intensity (applies to the gene expression panels)
    legend_loc = 'on data',                                                     # draw category labels directly on the clusters (only affects the categorical panel)
    legend_fontsize = 10,                                                       # shrink label text so it doesn't overlap too much with dense clusters
    legend_fontweight = 'bold',                                                 # bold labels so they're readable against the colored dots
    title = title                                                               # custom titles per panel
)
save_plot('08_umap_sst_markers.png')
plt.show()                                                                      # display the plot

<a name="sec-3-4-interp"></a>
### Interpretation: UMAP expression patterns

Sst is unevenly distributed, with visibly higher expression in some parts of the cluster that overlap with the vulnerable supertypes (red).

This gives us a solid baseline to see where and how strongly these markers are expressed across the Sst population as a whole, but so far we haven't accounted for disease severity at all. Next, we'll bring AD pathology into the picture, running a differential expression analysis to test whether these marker genes actually shift in expression between no and high-pathology donors.

<a name="sec-3-6"></a>
## 3.6 Differential gene expression in AD pathology

We will run a formal statistical test (Wilcoxon rank-sum) comparing marker gene expression between two groups within our Sst cells: donors with no AD pathology (`Not AD`) and donors with the highest severity (`High`).
Unlike a dot plot, this gives us an actual test statistic, effect size (log fold-change), and p-value for each gene, so we can ask whether an apparent difference is likely to be real rather than just visual noise.

We then visualize the results with a UMAP

Running this cell could take a bit (estimated ~ 3 minutes)

In [ ]:
# Identify the two conditions to compare
print(sst.obs['Overall AD neuropathological Change'].cat.categories)            # check the exact category labels available

condition_col = 'Overall AD neuropathological Change'                           # column that holds ADNC severity labels
ref_label = 'Not AD'                                                            # reference group label
max_label = 'High'                                                              # AD category

# uncomment if you want to change this
# markers2 = ['SST', 'GAD1', 'LHX6', 'ADARB2', 'SLC17A7']                       # same markers from earlier, these are the genes we'll compare

# Run differential expression between the two conditions within Sst cells only
sst_compare = sst[sst.obs[condition_col].isin([ref_label, max_label])].copy()   # .copy() avoids a "view" warning later

sc.tl.rank_genes_groups(
    sst_compare,                                                                # our two-condition Sst subset
    groupby = condition_col,                                                    # compare groups defined by ADNC status
    groups = [max_label],                                                       # test this group
    reference = ref_label,                                                      # against this one, so positive scores = higher in max AD
    method = 'wilcoxon',                                                        # statistical test used to rank genes
    key_added = 'rank_genes_adnc'                                               # store results under this key
)

# Pull out the DE stats for just our marker genes (not the full top-ranked list)
de_df = sc.get.rank_genes_groups_df(sst_compare, group=max_label, key='rank_genes_adnc')
marker_de_df = de_df[de_df['names'].isin(markers)]                              # filter down to only our marker genes
print(marker_de_df)                                                             # show fold-change/p-value stats

# Plot these marker genes on the UMAP side-by-side by condition
ref_adata = sst[sst.obs[condition_col] == ref_label]                            # cells from the reference group only
max_adata = sst[sst.obs[condition_col] == max_label]                            # cells from the max AD group only

# Fix the plot axis limits to the full Sst UMAP range so both panels are directly comparable
x_min, x_max = sst.obsm['X_umap'][:, 0].min(), sst.obsm['X_umap'][:, 0].max()
y_min, y_max = sst.obsm['X_umap'][:, 1].min(), sst.obsm['X_umap'][:, 1].max()

fig, axes = plt.subplots(len(markers), 2, figsize = (8, 4 * len(markers)))      # grid: rows = marker genes, cols = condition

row = 0                                                                         # counter to track which row of the grid we're on
for gene in markers:                                                            # loop through each marker gene
    sc.pl.umap(ref_adata, color = gene, cmap = 'viridis', ax = axes[row, 0],
               show = False, title = f'{gene} - {ref_label}')
    sc.pl.umap(max_adata, color = gene, cmap = 'viridis', ax = axes[row, 1],
               show = False, title = f'{gene} - {max_label}')
    for ax in axes[row]:                                                        # apply the same axis limits to both panels in this row
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
    row = row + 1                                                               # move to the next row for the next gene


plt.tight_layout()                                                              # clean up spacing between panels
save_plot('09_umap_de_notad_vs_high.png')
plt.show()                                                                      # display the full figure

<a name="sec-3-5-interp-de"></a>
### Interpretation: cell-level differential expression results

SST shows a large, significant decrease in the High group compared to Not AD, roughly a 2-fold drop. This supports the idea that SST expression is specifically reduced in Sst nuclei from donors with high AD pathology, consistent with the vulnerability described in the research paper.

This test is still run at the cell level rather than the donor level, so it doesn't fully account for the fact that nuclei from the same donor aren't independent samples. This is a caveat worth keeping in mind before treating this as definitive.

<a name="sec-3-6-pseudobulk"></a>
### 3.6.1 A more rigorous comparison: pseudobulk differential expression

The test above compared nuclei directly, treating each of the thousands of Sst nuclei as an independent sample. But nuclei from the same donor aren't truly independent, they share genetics, tissue handling, and sequencing batch. This can make results look more significant than they really are.

A more rigorous approach is called "pseudobulk". We average expression per donor first (one number per gene per donor), then compare donors between groups instead of comparing nuclei. This means each donor counts once, matching how many independent samples we actually have.

In [ ]:
# Compute each donor's average expression per marker gene
sst_compare = sst[sst.obs[condition_col].isin([ref_label, max_label])].copy()   # same two-condition subset as before

# Build a small dataframe: one row per cell, with donor ID, group,
# and expression values for our markers
expr_df = sc.get.obs_df(sst_compare, keys=['Donor ID', condition_col] + markers)

# Average expression per donor (collapses thousands of cells to one row per donor)
donor_avg = expr_df.groupby(['Donor ID', condition_col])[markers].mean().reset_index()
print(donor_avg)                                                                # one row per donor, showing their average expression of each marker gene

# Compare donor-level averages between the two groups, one gene at a time
results = []                                                                    # empty list to collect results for each gene

for gene in markers:                                                            # loop through each marker gene
    ref_values = donor_avg.loc[donor_avg[condition_col] == ref_label, gene]     # donor averages, reference group
    max_values = donor_avg.loc[donor_avg[condition_col] == max_label, gene]     # donor averages, max AD group

    stat, pval = mannwhitneyu(max_values, ref_values)                           # simple non-parametric test comparing two groups

    results.append({
        'gene': gene,
        'n_donors_ref': len(ref_values),
        'n_donors_max': len(max_values),
        'mean_ref': ref_values.mean(),
        'mean_max': max_values.mean(),
        'pval': pval
    })

pseudobulk_df = pd.DataFrame(results)                                           # turn our results into a clean table
print(pseudobulk_df)

A few notes on this approach:

We're skipping a multiple-testing correction here since we're only checking 5 genes and the goal is to illustrate the concept. For a real analysis testing many genes at once, you'd want to add a correction step (e.g., [`statsmodels.stats.multitest.multipletests`](https://www.statsmodels.org/devel/generated/statsmodels.stats.multitest.multipletests.html)) to avoid false positives from testing many hypotheses.

Finally, this is a simplified pseudobulk approach for teaching purposes. Single-nuclei pipelines often use more specialized tools built specifically for pseudobulk differential expression (like [`pydeseq2`](https://pydeseq2.readthedocs.io/en/stable/auto_examples/plot_pandas_io_example.html) or [`edgeR`](https://bioconductor.org/packages/release/bioc/vignettes/edgeR/inst/doc/intro.html), which model count data more carefully than a generic non-parametric test. The core idea, however, is the same: average per donor first, then compare donors, so each donor counts once instead of being split across many correlated nuclei.

In [ ]:
# Visualize donor-level ("pseudobulk") expression with one panel per gene

fig, axes = plt.subplots(1, len(markers), figsize = (4 * len(markers), 4))      # one panel per gene, side by side

col = 0                                                                         # counter to track which panel we're on
for gene in markers:                                                            # loop through each marker gene
    ax = axes[col]

    sns.boxplot(
        data = donor_avg,                                                       # our donor-level averages from the pseudobulk step
        x = condition_col,                                                      # group: Not AD vs High
        y = gene,                                                               # this gene's donor-average expression
        order = [ref_label, max_label],                                         # keep a consistent left-to-right order
        ax = ax,
        showfliers = False                                                      # hide default outlier markers since we'll show all points ourselves below
    )
    sns.stripplot(
        data = donor_avg,                                                       # same data, overlay dots
        x = condition_col,
        y = gene,
        order = [ref_label, max_label],
        ax = ax,
        color = 'black',                                                        # plot each donor as a black dot on top of the boxplot
        size = 5,
        alpha = 0.7
    )

    ax.set_title(gene)
    ax.set_xlabel('')                                                           # group labels are already clear from the x-axis ticks
    ax.set_ylabel('Mean expression per donor')

    col = col + 1                                                               # move to the next panel for the next gene

plt.tight_layout()
save_plot('10_pseudobulk_notad_vs_high.png')
plt.show()

<a name="sec-3-5-interp-pseudobulk"></a>
#### Interpretation: pseudobulk results

The pseudobulk analysis is in line with our earlier finding: SST expression is significantly lower in Sst nuclei from High AD pathology donors compared to Not AD donors (p = 0.039), after properly accounting for donor-level independence. The boxplot shows this isn't driven by a single outlier donor. Not AD donors are spread higher overall (median ~0.13) while High donors cluster tightly near zero (median ~0.05), with the separation visible across multiple donors in each group.

**Try it yourself:** We compared Not AD vs. High above. Try swapping the comparison to Low vs. High (or Intermediate vs. High) instead, by changing `ref_label` below, then re-running the donor-averaging and Mann-Whitney U test code. Does SST still show a significant difference, or does the effect look weaker between these less extreme severity groups?

In [ ]:
## Your code here



---

<a name="part-4"></a>
# Part 4: Vulnerable vs. Unaffected SST Supertypes

The rest of this notebook builds toward the question: **do the "vulnerable" Sst supertypes flagged by the paper appear different from the rest**? We'll build the comparison up in stages, deliberately including a couple of common pitfalls along the way since they're worth recognizing in your own analyses.


<a name="sec-4-1"></a>
## 4.1 Cell-proportion plots

We will re-create the proportion plot from above, but not highlight the vulnerable Sst categories.

In [ ]:
adnc_col = 'Overall AD neuropathological Change'                                # AD severity variable
color_by = 'Supertype'                                                          # column to break down proportions by

# build a cross-tab table with pandas (pd)
proportions = pd.crosstab(sst.obs[adnc_col],                                    # rows are AD severity categories
                          sst.obs[color_by],                                    # cols are Sst supertype
                          normalize = 'index')                                  # convert each row from raw counts into proportions that sum to 1

fig, ax = plt.subplots(figsize=(8, 6))                                          # fig setup/dims: single figure, 8x6 inches
# plotting code
proportions.plot(                                                               # plot the proportions df
    kind = 'bar',                                                               # pandas (matplotlib in background)
    stacked = True,                                                             # stacked bar
    ax = ax,                                                                    # draw on the ax setup above
    colormap = 'tab20',                                                         # tab20 palette, same one used in earlier Supertype UMAP
    width = 0.8,                                                                # bar width (leaves gaps between each)
)
# set plot parameters
ax.set_xlabel(adnc_col)                                                         # define x-axis label
ax.set_ylabel('Proportion of cells')                                            # define y-axis label
ax.set_title(f'{color_by} composition by {adnc_col}')                           # title based on cols chosen above
leg = ax.legend(bbox_to_anchor = (1.02, 1),
                loc = 'upper left', fontsize = 8,
                title = color_by)                                               # legend location/size

# highlight vulnerable supertypes in legend: red bold text
for text in leg.get_texts():                                                    # loop through each legend label
    if text.get_text() in vtypes:                                               # check if this supertype is one of our vulnerable ones
        text.set_color('red')                                                   # color vulnerable labels red
        text.set_fontweight('bold')                                             # bold them so they stand out

ax.tick_params(axis = 'x', rotation = 45)                                       # rotate x-axis ticks 45 degrees (non-overlapping)
ax.grid(False)                                                                  # remove background grid
plt.tight_layout()                                                              # adjust spacing to fit everything without cutoff
save_plot('11_sst_supertype_composition_by_adnc.png')
plt.show()                                                                      # show the figure

<a name="sec-4-1-interp"></a>
### Interpretation: supertype composition plot

Several vulnerable supertypes (red text in legend) decrease as a share of the Sst population with increasing AD severity (e.g. Sst_25), while some non-vulnerable supertypes make up a visibly larger share at High. As with our earlier proportion plot, this is built from nucleus-level counts rather than donor-level counts, so it's a visual hint of the pattern rather than a statistically validated one. Thus, it carries the same pseudoreplication caveat discussed earlier. A donor contributing unusually many or few nuclei can distort a bar's composition, so a donor-averaged version of this same plot (following the same approach we used for Subclass composition) would be a more rigorous way to confirm this pattern holds at the donor level.

<a name="sec-4-1-decline"></a>
### 4.1.1 Do vulnerable Sst supertypes decline with pathology?

Rather than plotting all 15 supertypes individually (which gets visually noisy fast), we'll go straight to comparing average supertypes based on two groups: vulnerable vs. unaffected. This will be plotted based on each group's mean proportion across ADNC severity, with a shaded band showing spread within each group.

In [ ]:
# Split supertype columns into vulnerable vs. unaffected groups
vuln_cols = proportions.columns.intersection(vulnerable_sst)                    # supertype columns flagged as vulnerable
unaff_cols = proportions.columns.difference(vulnerable_sst)                     # every other supertype column

# Get the order of ADNC categories
adnc_order = sst.obs[adnc_col].cat.categories.tolist()                          # category order as stored in the data

# Compute mean and standard deviation across supertypes, per ADNC stage
vuln_mean = proportions.loc[adnc_order, vuln_cols].mean(axis=1)                 # avg proportion across vulnerable supertypes, per stage
vuln_std = proportions.loc[adnc_order, vuln_cols].std(axis=1)                   # spread across vulnerable supertypes, per stage

unaff_mean = proportions.loc[adnc_order, unaff_cols].mean(axis=1)               # avg proportion across unaffected supertypes, per stage
unaff_std = proportions.loc[adnc_order, unaff_cols].std(axis=1)                 # spread across unaffected supertypes, per stage

# x-axis positions: 'Reference' is plotted as a standalone point,
# the remaining ADNC stages are connected as a progression line
x_positions = list(range(len(adnc_order)))                                      # one integer position per ADNC stage
x_reference = x_positions[0:1]                                                  # just 'Reference' (first in adnc_order)
x_progression = x_positions[1:]                                                 # every stage after 'Reference'

fig, ax = plt.subplots(figsize=(8, 6))                                          # single figure, 8x6 inches

# Vulnerable group: standalone Reference point + progression line
ax.scatter(x_reference, [vuln_mean['Reference']],
           color = 'crimson', s = 50, zorder = 3)                               # Reference plotted alone, not connected to the line
ax.plot(
    x_progression,
    vuln_mean[adnc_order[1:]],                                                  # mean proportion at each non-Reference stage
    marker = 'o',                                                               # dot marker at each stage
    color = 'crimson',                                                          # vulnerable group's color, matches earlier plots
    linewidth = 2.5,                                                            # line thickness
    label = 'Vulnerable (mean)',                                                # legend entry
)
ax.fill_between(
    x_progression,
    (vuln_mean - vuln_std)[adnc_order[1:]],                                     # lower bound of the shaded band
    (vuln_mean + vuln_std)[adnc_order[1:]],                                     # upper bound of the shaded band
    color = 'crimson',                                                          # match the vulnerable line color
    alpha = 0.15,                                                               # light shading so the line underneath stays visible
)

# Unaffected group: standalone Reference point + progression line
ax.scatter(x_reference, [unaff_mean['Reference']],
           color = 'gray', s = 50, zorder = 3)                                  # Reference plotted alone, not connected to the line
ax.plot(
    x_progression,
    unaff_mean[adnc_order[1:]],                                                 # mean proportion at each non-Reference stage
    marker = 'o',                                                               # dot marker at each stage
    color = 'gray',                                                             # unaffected group's color, matches earlier plots
    linewidth = 2.5,                                                            # line thickness
    label = 'Unaffected (mean)',                                                # legend entry
)
ax.fill_between(
    x_progression,
    (unaff_mean - unaff_std)[adnc_order[1:]],                                   # lower bound of the shaded band
    (unaff_mean + unaff_std)[adnc_order[1:]],                                   # upper bound of the shaded band
    color = 'gray',                                                             # match the unaffected line color
    alpha = 0.15,                                                               # light shading so the line underneath stays visible
)

# Axis labels, ticks, and styling
ax.set_xticks(x_positions)                                                      # one tick per ADNC stage
ax.set_xticklabels(adnc_order, rotation = 45, ha = 'right')                     # label each tick with its stage name
ax.set_xlabel(adnc_col)                                                         # x-axis label
ax.set_ylabel('Mean proportion of cells per supertype')                         # y-axis label
ax.set_title('Average Sst supertype proportion: Vulnerable vs. Unaffected')     # plot title
ax.legend(loc = 'upper left')                                                   # show the two group labels
ax.grid(False)                                                                  # remove background grid

plt.tight_layout()                                                              # adjust spacing to fit everything without cutoff
save_plot('12_vulnerable_vs_unaffected_trend.png')
plt.show()                                                                      # display the plot

<a name="sec-4-1-interp-trend"></a>
#### Interpretation: vulnerable vs. unaffected trend

For the vulnerable supertypes, their average share of the Sst population declines relative to the unaffected group as ADNC severity increases, though there's spread within each group (shown by the shaded bands). This is a group-level trend, not a claim that every single vulnerable supertype behaves identically. Next, we'll test this more directly with differential expression: do vulnerable and unaffected Sst supertypes actually express different genes?

<a name="sec-4-2"></a>
## 4.2 Differential expression: Vulnerable vs. Unaffected

First, we'll label every Sst cell as "Vulnerable" or "Unaffected" based on its supertype, using the `vulnerable_sst` list from earlier.

In [ ]:
# tag each cell as vulnerable or unaffected (vulnerable_sst was defined above)
is_vulnerable = sst.obs['Supertype'].isin(vulnerable_sst)                       # True/False for each cell
sst.obs['vulnerability'] = 'Unaffected'                                         # start by labeling every cell as unaffected
sst.obs.loc[is_vulnerable, 'vulnerability'] = 'Vulnerable'                      # relabel the vulnerable ones

As we found earlier, the Reference cohort is separate than the SEA-AD donors, so we'll exclude Reference cells before running differential expression.

In [ ]:
# remove reference category here
sst_no_ref = sst[sst.obs['Overall AD neuropathological Change'] != 'Reference'].copy()

Running this cell will take a bit (estimated ~ 5 minutes)

In [ ]:
# differential expression: vulnerable vs. unaffected Sst supertypes
sc.tl.rank_genes_groups(
    sst_no_ref,                                                                 # use the no ref dataset
    groupby = 'vulnerability',                                                  # column that defines the two groups to compare
    groups = ['Vulnerable'],                                                    # test this group
    reference = 'rest',                                                         # against everything else (equivalent to 'Unaffected' here, but computes both pct columns)
    method = 'wilcoxon',                                                        # statistical test used for differential expression
    pts = True                                                                  # also report the percentage of cells expressing each gene
)

# pull the results into a dataframe for the 'Vulnerable' group
vuln_de = sc.get.rank_genes_groups_df(sst_no_ref, group = 'Vulnerable')

# show the top 20 genes, ranked by adjusted p-value
vuln_de.sort_values('pvals_adj').head(20)

**Why NGF and MME?** These two genes are highlighted throughout this section because the paper specifically plots them in [Figure 5i](https://www.nature.com/articles/s41593-024-01774-5/figures/5). `NGF` and `MME` expression declines specifically in vulnerable Sst supertypes as pathology progresses, while unaffected supertypes hold comparatively steady. This trajectory-based result is why these two genes were picked as points of interest here, even though our own test below asks a related but different question.

Our `rank_genes_groups` test compares Vulnerable vs. Unaffected cells pooled across all ADNC stages. It asks: is average expression different between the two groups overall?

The paper's Figure 5i instead asks: does expression change over the course of pathology, and does that change differ by group?

So a gene can show a modest or even different-direction effect in our simple two-group comparison while still showing a clear declining trend specifically in vulnerable types once pathology stage is taken into account.


In [ ]:
# pull out the differential expression results for just these two genes of interest (reported in paper)
vuln_de[vuln_de['names'].isin(['NGF', 'MME'])]

<a name="sec-4-3"></a>
## 4.3 Volcano plots

Our differential expression table (`vuln_de`) has one row per gene, and for each gene we care about two things at once: how *big* the difference is between Vulnerable and Unaffected (the fold change), and how *confident* we are that the difference is real (the adjusted p-value). A volcano plot puts both on a single scatter plot: fold-change on the x-axis and significance on the y-axis. The genes that are both strongly changed *and* statistically significant stand out in the upper corners, which is where the plot gets its name.

We'll define one reusable plotting function below.

In [ ]:
#@title Volcano plot function
# A reusable function for drawing a volcano plot, so we don't repeat this code twice
def plot_volcano(df, title, y_label, y_max = None, highlight_genes = ['NGF', 'MME'], save_name = None):
    # significance/direction thresholds
    padj_cutoff = 0.05
    lfc_cutoff = 1                                                              # log2FC threshold, adjust as desired

    # label each gene based on which side of both thresholds it falls on
    def classify(row):
        if row['pvals_adj'] < padj_cutoff and row['logfoldchanges'] >= lfc_cutoff:
            return 'Up in Vulnerable'
        elif row['pvals_adj'] < padj_cutoff and row['logfoldchanges'] <= -lfc_cutoff:
            return 'Up in Unaffected'
        else:
            return 'Not significant'

    df = df.copy()                                                              # work on a copy so we don't modify the caller's dataframe
    df['category'] = df.apply(classify, axis = 1)                               # apply the classifier to every row (gene)

    # color to use for each category in the plot
    color_map = {
        'Up in Vulnerable': 'crimson',
        'Up in Unaffected': 'steelblue',
        'Not significant': 'lightgray'
    }

    fig, ax = plt.subplots(figsize = (9, 7))

    # plot each category separately so it gets its own color and legend entry
    for cat, color in color_map.items():
        subset = df[df['category'] == cat]
        ax.scatter(subset['logfoldchanges'], subset['neg_log10_padj'], c = color, s = 10, alpha = 0.6, label = cat)

    # threshold reference lines
    ax.axhline(-np.log10(padj_cutoff), color = 'black', linestyle = '--', linewidth = 0.8)
    ax.axvline(lfc_cutoff, color = 'black', linestyle = '--', linewidth = 0.8)
    ax.axvline(-lfc_cutoff, color = 'black', linestyle = '--', linewidth = 0.8)

    # highlight and label the requested genes (defaults to NGF/MME, but any gene list can be passed in)
    # xytext is staggered by index (6, 6), (6, 18), (6, 30), so labels for
    # points that sit close together (e.g. several near the same p-value cap)
    # don't land on top of each other and become unreadable
    for i, gene in enumerate(highlight_genes):
        row = df[df['names'] == gene]
        if not row.empty:
            x, y = row['logfoldchanges'].values[0], row['neg_log10_padj'].values[0]
            ax.scatter(x, y, c = 'black', s = 60, zorder = 5, edgecolor = 'yellow', linewidth = 1.5)
            ax.annotate(gene, (x, y), textcoords = "offset points", xytext = (6, 6 + 12 * i), fontsize = 10, fontweight = 'bold')

    # only clip the y-axis if the caller asked for a fixed ceiling
    if y_max is not None:
        ax.set_ylim(0, y_max + 5)

    # axis labels, title, legend, and final styling
    ax.set_xlabel('log2 fold change (Vulnerable vs. Unaffected)')
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.legend(loc = 'upper left', fontsize = 9)
    ax.grid(False)

    plt.tight_layout()
    if save_name is not None:
        save_plot(save_name)
    plt.show()


<a name="sec-4-3-step1"></a>
### Step 1

With the function defined, let's compute -log10(adjusted p-value) for every gene and plot it, without worrying yet about edge cases.

In [ ]:
df = vuln_de.copy()
df['neg_log10_padj'] = -np.log10(df['pvals_adj'].replace(0, np.nextafter(0, 1)))   # avoid log(0)

plot_volcano(df,
             title = 'Vulnerable vs. Unaffected volcano plot',
             y_label ='-log10(adjusted p-value)',
             save_name = '13_volcano_raw.png')

**Issue 1: The flat "ceiling" row at the top**

That flat line of points all sitting at the same height (~y=325) is an artifact of how we handled `pvals_adj == 0`. `np.nextafter(0, 1)` returns the smallest possible floating-point number. A tiny number, the same for every gene that hit exactly 0. So every gene with `pvals_adj` rounding to zero (which with large cell counts is common) all get identically mapped to the same, large -log10 value, creating that flat row instead of a smooth spread.

Fix: instead of using the smallest possible float, cap at something based on the smallest actual non-zero p-value in the data, so the "floor" is realistic rather than artificially extreme

**Issue 2: Extreme x-axis outliers with near-zero significance**

Those scattered points way out at ±12 to ±18 sitting right at the bottom (y≈0, not significant) are a  `scanpy` log-fold-change artifact: genes with very low expression in both groups (e.g., detected in only a handful of nuclei total) can produce unstable fold-change estimates purely from small-number noise, even though they're nowhere close to statistically significant. This is a well-known pitfall in DE fold-change calculations generally.

Fix: filter out genes with negligible expression in both groups before plotting, using the `pct_nz_group`/`pct_nz_reference` columns in the data

<a name="sec-4-3-step2"></a>
### Step 2

Putting both fixes together to give a more interpretable plot:
1. Dropping genes with negligible expression in both groups
2. Capping -log10(adjusted p-value) at a realistic ceiling instead of relying on the smallest possible float



In [ ]:
df = vuln_de.copy()                                                             # work on a copy so we don't modify the original DE results
df = df[(df['pct_nz_group'] > 0.05) | (df['pct_nz_reference'] > 0.05)]          # drop near-zero-expression genes

CAP = 50                                                                        # ceiling for -log10(adjusted p-value) instead of using the smallest possible float

# avoid log(0) issues for genes with pvals_adj == 0, then cap at a realistic ceiling
df['neg_log10_padj'] = -np.log10(df['pvals_adj'].replace(0, np.nextafter(0, 1)))
df['neg_log10_padj'] = df['neg_log10_padj'].clip(upper = CAP)

# draw the volcano plot using the shared function, with the fixed y-axis ceiling
plot_volcano(df,
            title = 'Vulnerable vs. Unaffected - adjusted volcano plot',
            y_label = f'-log10(adjusted p-value), capped at {CAP}',
            y_max = CAP,
            save_name = '14_volcano_fixed.png')


<a name="sec-4-3-interp"></a>
#### Interpretation: volcano plot findings

The volcano plot shows a clear asymmetry: many more genes are strongly upregulated in Vulnerable supertypes (red) than in Unaffected ones (blue), with larger fold changes, suggesting vulnerable Sst supertypes are transcriptionally distinct rather than just less abundant. MME stands out as one of the most significant and strongly upregulated genes in the Vulnerable group, while NGF is only marginally significant with a small fold change, falling short of the log2FC cutoff used to flag genes as strongly differentially expressed. The dense cloud of gray points near log2FC = 0 represents the majority of genes showing no meaningful difference, which is expected.

Keep in mind this test was run at the nucleus level rather than pseudobulk, so with large nuclei counts per donor, p-values can look more extreme than they would with donor-level averaging. The fold-change magnitudes are the more directly interpretable signal here.

Further, this is different than the published paper where MME expression declines in vulnerable Sst supertypes as pathology worsens ([Figure 5i](https://pmc.ncbi.nlm.nih.gov/articles/PMC11614693/figure/Fig5/)). That is a different comparison than what we did here. The volcano plot here pools nuclei across all AD stages and asks whether Vulnerable and Unaffected nuclei differ in average expression overall; MME being higher here just means vulnerable nuclei express more MME than unaffected nuclei on average across the whole dataset.

**Try it yourself:** So far every plot has only highlighted `NGF` and `MME`. Try highlighting the top 5 most significant genes overall instead:

1. Sort `df` by `pvals_adj` and pull the top 5 gene names into a list
  - look back at how `vuln_de.sort_values('pvals_adj').head(20)` worked in [Section 4.2](#sec-4-2))
  - you will also need to add `['names'].tolist()` to that line
2. Call `plot_volcano(...)` again, passing that list in as `highlight_genes` (it defaults to `['NGF', 'MME']`, but you can pass any list of gene names).

You don't need to touch anything inside `plot_volcano` itself because `highlight_genes` was built as a parameter to swap.


In [ ]:
## Your code here - edit ...

# get the 5 gene names with the smallest adjusted p-value (i.e. most significant)
top5_genes = ...
print(top5_genes)


In [ ]:
# re-draw the fixed volcano plot, but highlight the top 5 genes instead of NGF/MME
plot_volcano(
    df,
    title = 'Vulnerable vs. Unaffected - top 5 significant genes highlighted',
    y_label = f'-log10(adjusted p-value), capped at {CAP}',
    y_max = CAP,
    highlight_genes = top5_genes,
    save_name = '15_volcano_top5.png'
)

**Note:** Some (or all) of these top-5-by-p-value genes may show up in gray rather than red/blue. The color coding requires *both* a significant p-value *and* a fold-change above the cutoff. `top5_genes` was selected by p-value alone. A gene can be statistically significant (especially at the nucleus level, with thousands of nuclei as "samples") while still having a small, biologically modest fold-change. Statistical significance and effect size are two different things, and this is a good example of where they diverge.

---

<a name="part-5"></a>
# Part 5: Try it yourself by repeating this analysis for a different subclass

A recap of what we did for Sst: picked marker genes, subset the data, checked composition across AD severity, compared vulnerable vs. unaffected supertypes, and built a volcano plot.

Gabitto, Travaglini, et al. (2024) found that **Pvalb** interneurons are also AD-vulnerable, but on a different timeline than Sst (declining sharply only at high pathology, rather than early and gradually). Here's the outline for redoing this analysis on **Pvalb**, or any other subclass you're curious about. No new code is provided, this is a chance to reuse and adapt what we've already built.

1. **Subset the data:** Pull out just the Pvalb cells from `adata_meta`, the way we built `sst` back in [Section 3.2](#sec-3-2).
2. **Identify marker genes:** Look up canonical Pvalb markers (e.g., `PVALB`, `GAD1`, `LHX6`) and a negative control (`SLC17A7` still works), then run the same sanity-check dot plot from [Section 3.3](#sec-3-3).
3. **Check composition across ADNC:** Reuse the stacked bar chart and aggregate vulnerable vs. unaffected trend plot from [Part 4](#part-4), swapping in Pvalb's own vulnerable supertype list from [Section 3.2.1](#sec-3-2-vtypes)
4. **Run differential expression.** Reuse the vulnerable vs. unaffected Wilcoxon test and pseudobulk approach from [Section 4.2](#sec-4-2).
5. **Build a volcano plot.** Reuse the `plot_volcano` function from [Section 4.3](#sec-4-3) directly, it doesn't depend on anything Sst-specific.

As you go, keep an eye out for whether Pvalb's pattern actually looks different from Sst's. The paper's finding was that Pvalb declines sharply only at high pathology, rather than early and gradually like Sst. Does your own analysis support that?

In [ ]:
# Your code here

---
<a name="sec-6"></a>
# Part 6: Saving the data outside of this Colab session

Anything in `/content` is **deleted when the runtime ends** (closing the tab, idle timeout, or *Runtime > Disconnect and delete runtime*). To keep the file, save it somewhere else before then.

In this session, we accessed or created the following files:
- `/content/plots`: contains individual png images of plots from the notebook
  - These images will still exist in *this notebook* if you saved it for returning to it in the future
- `/content/SEAAD_MTG_2024_RNAseq.h5ad`: is the large (~34 GB) data file
- `/content/SEAAD_MTG_SST_Subset_2024.h5ad`: was an optional saving step of the Sst data subset


There are several options for saving:

### <u>Option 1</u>: Download from the file browser
1. Click the 📁 **folder icon** in the left sidebar
2. Find the file (it's at the top level of `/content`). Click the refresh icon if it doesn't show up
3. Right-click the file (3 dots appear when hovering the mouse on the right side) and choose **Download**

### <u>Option 2</u>: Download the file directly from the AWS bucket
You can select the brain region folder and file from the website directly. Clicking on the name of a file will start a download locally to your computer: https://sea-ad-single-cell-profiling.s3.amazonaws.com/index.html


# Questions or Looking for Resources?

* Search and post on the community forum: https://community.brain-map.org/tag/sea-ad/95

# Credits

* Thanks to colleagues at the Allen Institute for
sharing resources and data from this project.
* This notebook was made as part of the SPOCTT (Scientific and Public Outreach of Cell Type Taxonomies) Initiative
* Created by: Theresa H. McKim, PhD (tmckim@unr.edu)
* Shared under CC BY 4.0: you may reuse and adapt this notebook with attribution.